In [ ]:
from datasets import load_dataset
imdb_dataset= load_dataset("stanfordnlp/imdb")

In [ ]:
import random
from collections import Counter
from typing import List, Tuple
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Dataset

In [ ]:
SEED = 42
EMBED_DIM = 100
WINDOW_SIZE = 2
NEGATIVE_SAMPLES = 5
BATCH_SIZE = 512
EPOCHS = 5
LR = 0.002 # learning rate
NUM_DOCUMENTS = 100 # 읽을 문서 수
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
"cpu")

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

# **문서를 단어 단위로 쪼개기**

In [ ]:
def read_corpus() -> List[List[str]]:
    imdb_dataset = load_dataset("stanfordnlp/imdb")
    files: List[str] = imdb_dataset["train"]["text"][:NUM_DOCUMENTS]
    print(f"files_example = {files[1]}")
    # return 구문 상세화~
    # output = []
    # for f in files:
    #   f: str
    #   inner_1 = []
    #   for w in f.split():
    #     inner_1.append(w.lower())
    #   output.append(inner_1)
    # files = output
    return [[w.lower() for w in f.split()] for f in files]


tokens_list = read_corpus()
print(f"#(documents): {len(tokens_list)}")
print("The number of tokens in the 1st document:", len(tokens_list[0]))
print(f"imdb_corpus[0][:20] = {tokens_list[0][:20]}")

# 단일 토큰 시퀀스로 flatten
all_tokens: List[str] = [tok for doc in tokens_list for tok in doc]

files_example = "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we're treated to the site of Vincent Gallo's throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, a

# **vocab생성**

In [ ]:
def build_vocab(tokens_list: List[List[str]], tokens: List[str]) -> Tuple[dict,dict, List[List[int]]]:
  word2idx = {}
  idx2word = {}
  i=0
  #word2idx["AAAA"] = 1
  for token in tokens:
    if token not in word2idx:
      word2idx[token] = i
      idx2word[i] = token
      i += 1
  # idx2word = {v: k for k, v in word2idx.items()}
  indexed_docs : list[list[int]] = [[word2idx[token] for token in tokens] for tokens in tokens_list]
  return word2idx, idx2word, indexed_docs

word2idx, idx2word, indexed_docs = build_vocab(tokens_list, all_tokens)
vocab_size = len(word2idx)
print(f"어휘 수: {vocab_size}")
print(indexed_docs[0][:20])

어휘 수: 6008
[0, 1, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 15, 17]


# **pair 생성**

In [ ]:
def build_skipgram_pairs_docs(indexed_docs: list[list[int]], window_size:int =2) -> list[Tuple[int, int]]:
  output = []
  print("idx in doc :", indexed_docs[0])
  for doc in  indexed_docs:
    for c_i , c in enumerate(doc):
      left = max(0,c_i - window_size)
      right = min(c_i + window_size, len(doc) - 1)
      for j in range(left, right + 1):
        if j != c_i:
          output.append((c, doc[j]))
  return output

pairs = build_skipgram_pairs_docs(indexed_docs, window_size=1)
print(pairs[:10])

idx in doc : [0, 1, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 15, 17, 18, 19, 20, 21, 0, 22, 23, 13, 24, 18, 15, 17, 25, 26, 27, 28, 29, 15, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 9, 40, 41, 42, 0, 43, 44, 32, 45, 34, 46, 47, 48, 49, 50, 51, 52, 53, 38, 54, 55, 56, 57, 58, 59, 60, 61, 32, 62, 63, 64, 65, 66, 67, 20, 68, 64, 61, 32, 69, 70, 71, 32, 72, 73, 74, 9, 75, 76, 77, 11, 78, 79, 80, 66, 81, 82, 83, 84, 85, 11, 86, 87, 88, 89, 83, 20, 11, 90, 91, 20, 92, 93, 94, 88, 95, 96, 9, 97, 66, 98, 99, 76, 100, 64, 101, 102, 103, 70, 56, 104, 105, 88, 106, 107, 48, 108, 109, 110, 66, 0, 2, 3, 51, 13, 111, 112, 113, 34, 17, 41, 114, 115, 11, 102, 88, 116, 117, 118, 119, 88, 120, 121, 122, 123, 124, 125, 126, 127, 73, 128, 129, 130, 131, 5, 132, 133, 134, 15, 135, 20, 136, 102, 88, 116, 118, 38, 137, 138, 20, 55, 139, 122, 140, 141, 142, 98, 143, 32, 144, 145, 146, 147, 148, 44, 102, 117, 20, 149, 150, 48, 151, 152, 153, 11, 154, 46, 11, 155, 13, 156, 102, 157, 20, 11, 158, 51,

# **스무딩해서 단어 별 확률 분표 표시**

In [ ]:
# @title
def make_unigram_probs(indexed_docs:list[list[int]], vocab_size: int, power: float= 0.75) -> np.ndarray:
    freqs= np.zeros(vocab_size, dtype=np.float64)
    for doc in indexed_docs:
        for i in doc:
          freqs[i] += 1

    probs= freqs**power
    probs/=probs.sum()
    return probs

probs = make_unigram_probs(indexed_docs)
print("단어별 확률 분포:")
for i, p in enumerate(probs):
    print(f"단어 {i}: {p:.4f}")

print("\n확률의 합:", probs.sum())

NameError: name 'np' is not defined

In [ ]:
class SkipGramDataset(Dataset):
    def __init__(self, pairs: List[Tuple[int, int]], probs: np.ndarray, vocab_size: int, k: int):
      self.pairs= pairs
      self.probs= probs
      self.vocab_size= vocab_size
      self.k= k

    def __len__(self):
      return len(self.pairs)

    def __getitem__(self, idx):

        center, pos= self.pairs[idx]

        negs= np.random.choice(self.vocab_size, size=self.k, replace=True, p=self.probs)
        return(
        torch.tensor(center),
        torch.tensor(pos),
        torch.tensor(negs),
        )

In [ ]:
dataset= SkipGramDataset(pairs, probs, vocab_size, NEGATIVE_SAMPLES)
print(f"Thefirst example: {dataset[1]}")

Thefirst example: (tensor(1), tensor(0), tensor([2142,   70,  366,  664, 1070]))


# *Skip-gram 모델 도입*

In [ ]:
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.in_embed.weight, a=-0.5/embed_dim, b=0.5/embed_dim)
        nn.init.uniform_(self.out_embed.weight, a=-0.5/embed_dim, b=0.5/embed_dim)

    def forward(self, center, pos, neg):
      # B: mini-batch size, K: #(negative samples), D: self.embed_dim
      # cent: (B), pos: (B), neg: (B, K)
      cent= self.in_embed(cent) # (B, D)
      pos= self.in_embed(pos) # (B, D)
      neg= self.in_embed(neg) # (B, K, D)
      pos_mult= (cent* pos) # (B, D)
      pos_inner= pos_mult.sum(dim=-1) # (B)
      cent_2= cent.reshape(-1, 1, self.embed_dim) # (B, 1, D)
      neg_mult= cent_2* neg# (B, K, D)
      neg_inner= neg_mult.sum(dim=-1) # (B, K)
      pos_loss= -nn.functional.logsigmoid(pos_inner) # (B)
      neg_loss= -nn.functional.logsigmoid(neg_inner).sum(dim=-1) # (B)
      return(pos_loss+neg_loss).mean()


In [ ]:
def train():
